## Event ROI Generator — Plan → Batch Query → Synthesize

This notebook implements an agent-powered pipeline that generates standardized post-event ROI reports by:

1. **PLAN** (~2 sec) — An orchestrator LLM parses the user request, resolves the event name, and generates 5–8 wide Genie questions (each covering multiple metrics).
2. **BATCH QUERY** (~15–20 sec) — All questions are sent to the MARDI Genie Space in parallel. Raw tabular data is collected into a compact ~5–10K token data packet.
3. **SYNTHESIZE** (~5–10 sec) — A single LLM call receives the full data packet + report template + metric definitions, producing a coherent 7-slide narrative.

**Data Sources:**
| Domain | Table |
|---|---|
| Registration & Attendance | `main.marketing_lakehouse.rpt_event_performance` |
| Pipeline Attribution | `main.marketing_lakehouse.fact_funnel_for_analyst` |
| Cost & Spend | `main.marketing_lakehouse.gold_roi_attribution_spend` |

**Core Metrics:** Cost per Lead · Pipeline Generated · fPOI (Forecasted Pipeline on Investment)

## Gemini API Option

This notebook uses **Databricks-hosted LLMs** (Claude / GPT) via .

A production **React UI** has been built alongside this notebook at:
- **Frontend**:  — React + TypeScript + Tailwind
- **Backend API**:  — FastAPI + Gemini + MARDI Genie

To use **Google Gemini** instead (outside Databricks), set  in 
and run the FastAPI server. The UI will automatically use Gemini for Planning and Synthesis
while still querying MARDI Genie for data.

### Key Improvements in the API
- Real-time SSE streaming of pipeline progress
- Editable 7-slide outline before PPT generation  
- One-click PowerPoint download (python-pptx)
- Demo mode with sample AI Days data (no Databricks needed)

In [0]:
%pip install -U -qqqq databricks-openai
dbutils.library.restartPython()

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
%python
import time
import json
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from databricks.sdk import WorkspaceClient
from databricks_openai import DatabricksOpenAI

# ---------------------------------------------------------------------------
# Genie Space Configuration
# ---------------------------------------------------------------------------
MARDI_SPACE_ID = "01f137f8ed74188983eabc78abf23274"

# ---------------------------------------------------------------------------
# LLM Configuration (Databricks Foundation Model APIs)
# ---------------------------------------------------------------------------
LLM_ENDPOINTS = {
    "claude": "databricks-claude-sonnet-4-5",
    "gpt":    "databricks-gpt-4.1",
}
DEFAULT_LLM = "claude"

# ---------------------------------------------------------------------------
# Metric Definitions (from the Event ROI Generator spec)
# ---------------------------------------------------------------------------
METRIC_DEFINITIONS = """
## Metric Definitions
- **fPOI (Forecasted Pipeline on Investment)** = SUM(u2_ARR) / SUM(allocated_spend)
  Formula: IF(SUM(allocated_spend) = 0, 0, SUM(u2_ARR) / SUM(allocated_spend))
- **Attendance Rate** = Attended / Registered × 100
- **U2** = Pipeline Created stage
- **U3 / QRR** = Qualified Pipeline stage

## Attribution Rules
- Use existing MTA from gold tables — do NOT re-derive attribution
- Pipeline numbers mature over 30–60 days post-event
- Always use pre-computed registered/attended from rpt_event_performance
- Roll up at parent_program_name level for multi-program events
- Join logic: parent_program_name ↔ program for spend linkage
"""

# ---------------------------------------------------------------------------
# Report Template (7 slides)
# ---------------------------------------------------------------------------
REPORT_TEMPLATE = """
## Report Structure (7 Slides)

### Slide 1: Executive Summary
KPI headlines: Total Pipeline, Attendance Rate

### Slide 2: Registration & Attendance
Registered count, Attended count, Attendance rate, In-person vs virtual breakdown

### Slide 3: Pipeline Funnel
NNN → MQL → U2 → U3 progression with conversion rates at each stage

### Slide 4: Account Segment Analysis
Business unit breakdown, spend tier, DBX customer vs prospect

### Slide 5: Cost & ROI Analysis
Blank for now

### Slide 6: YoY Comparison
Blank for now

### Slide 7: Cross-Event Comparison
Blank for now
"""

print("Configuration loaded.")

Configuration loaded.


In [0]:
%python
# ---------------------------------------------------------------------------
# Genie Client — adapted from MARDI notebook with parallel query support
# ---------------------------------------------------------------------------

class GenieClient:
    """Client to interact with MARDI (Databricks Genie Space).
    
    Supports both single queries and parallel batch queries.
    Returns raw structured data (columns + rows) for downstream processing.
    """

    def __init__(self, space_id: str = MARDI_SPACE_ID):
        self.w = WorkspaceClient()
        self.space_id = space_id

    def ask(self, question: str) -> dict:
        """Send a single question to Genie and return structured response."""
        print(f"\U0001f9de Asking MARDI: \"{question}\"")

        # Start conversation and wait for completion
        message = self.w.genie.start_conversation_and_wait(
            space_id=self.space_id,
            content=question,
        )

        conversation_id = message.conversation_id
        message_id = message.message_id

        # Fetch the full message with attachments
        msg = self.w.genie.get_message(
            space_id=self.space_id,
            conversation_id=conversation_id,
            message_id=message_id,
        )

        result = {
            "question": question,
            "conversation_id": conversation_id,
            "message_id": message_id,
            "text_response": "",
            "sql_query": None,
            "data_columns": [],
            "data_rows": [],
            "status": str(msg.status),
            "error": None,
        }

        # Extract text, SQL, and data from attachments
        if msg.attachments:
            for att in msg.attachments:
                if getattr(att, "text", None) and getattr(att.text, "content", None):
                    result["text_response"] = att.text.content

                if getattr(att, "query", None) and getattr(att.query, "query", None):
                    result["sql_query"] = att.query.query

                # Extract raw tabular data from query results
                att_id = getattr(att, "attachment_id", None) or getattr(att, "id", None)
                if att_id:
                    try:
                        qr = self.w.genie.get_message_attachment_query_result(
                            space_id=self.space_id,
                            conversation_id=conversation_id,
                            message_id=message_id,
                            attachment_id=att_id,
                        )
                        stmt = qr.statement_response
                        if stmt and stmt.manifest and stmt.manifest.schema and stmt.manifest.schema.columns:
                            result["data_columns"] = [c.name for c in stmt.manifest.schema.columns]
                        if stmt and stmt.result:
                            if stmt.result.data_array:
                                result["data_rows"] = stmt.result.data_array
                            elif stmt.result.external_links:
                                for link in stmt.result.external_links:
                                    if getattr(link, "external_link", None):
                                        resp = requests.get(link.external_link)
                                        if resp.ok:
                                            try:
                                                result["data_rows"] = json.loads(resp.text)
                                            except Exception:
                                                result["data_rows"] = [[resp.text[:2000]]]
                    except Exception as e:
                        print(f"\u26a0\ufe0f Could not fetch query result for attachment {att_id}: {e}")

        print(f"  \u2192 Status: {result['status']} | Columns: {len(result['data_columns'])} | Rows: {len(result['data_rows'])}")
        return result

    def batch_ask(self, questions: list[str], max_workers: int = 5) -> list[dict]:
        """Send multiple questions to Genie in parallel.
        
        Returns a list of results in the same order as the input questions.
        Failed queries return partial results with error info instead of raising.
        """
        print(f"\n\U0001f680 Sending {len(questions)} questions to MARDI in parallel...")
        start_time = time.time()
        results = [None] * len(questions)

        def _ask_with_index(idx_question):
            idx, question = idx_question
            try:
                return idx, self.ask(question)
            except Exception as e:
                print(f"\u274c Query {idx + 1} failed: {e}")
                return idx, {
                    "question": question,
                    "text_response": "",
                    "sql_query": None,
                    "data_columns": [],
                    "data_rows": [],
                    "status": "FAILED",
                    "error": str(e),
                }

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = executor.map(_ask_with_index, enumerate(questions))
            for idx, result in futures:
                results[idx] = result

        elapsed = time.time() - start_time
        successes = sum(1 for r in results if r and r.get("status") != "FAILED")
        print(f"\n\u2705 Batch complete: {successes}/{len(questions)} succeeded in {elapsed:.1f}s")
        return results


print("GenieClient ready.")

GenieClient ready.


In [0]:
%python
# ---------------------------------------------------------------------------
# Phase 1: PLAN — Orchestrator LLM generates wide Genie questions
# ---------------------------------------------------------------------------

PLANNER_SYSTEM_PROMPT = """
You are an event analytics planner for Databricks marketing. Your job is to
take a user request about an event report and produce 5–8 wide, data-rich
questions for a Genie analytics agent (MARDI) that has access to these tables:

- rpt_event_performance: registration, attendance, NNN, MQL counts by event/child program, pipeline attribution — U2 (created), U3/QRR (qualified), amounts

Rules:
1. Each question should request MULTIPLE metrics in one query (wide, not narrow).
2. Group by child_program or relevant dimension where useful.
3. Always include one question for registration + attendance totals.
4. Always include one question for pipeline funnel (NNN → MQL → U2 → U3).
5. Always include one question for spend / cost data.
6. Include a YoY comparison question if a prior-year event likely exists.
7. Include a question about account segmentation if relevant.
8. Return ONLY a JSON array of question strings. No commentary.

Example output:
[
  "Show me total registered, attended, NNN, and MQL for NRF 2026 grouped by child program",
  "What is the U2 pipeline amount, U2 ARR, U3 pipeline amount, and deal count for NRF 2026",
  "Show the total allocated spend and spend breakdown by category for NRF 2026",
  "What were the registration, attendance, and pipeline numbers for NRF 2025 for YoY comparison",
  "Show unique accounts, DBX customer vs prospect breakdown for NRF 2026 attendees"
]
"""


def plan_questions(user_request: str, llm: str = DEFAULT_LLM) -> list[str]:
    """Phase 1: Use the orchestrator LLM to generate wide Genie questions.
    
    Args:
        user_request: Natural language request like "Generate report for NRF 2026"
        llm: Which Databricks-hosted LLM to use ("claude" or "gpt")
    
    Returns:
        List of 5-8 wide question strings for Genie
    """
    print(f"\n{'='*70}")
    print(f"PHASE 1: PLAN")
    print(f"{'='*70}")
    print(f"User request: \"{user_request}\"")
    start_time = time.time()

    endpoint = LLM_ENDPOINTS[llm.lower()]
    client = DatabricksOpenAI()

    response = client.chat.completions.create(
        model=endpoint,
        messages=[
            {"role": "system", "content": PLANNER_SYSTEM_PROMPT},
            {"role": "user", "content": user_request},
        ],
        temperature=0.2,
        max_tokens=1500,
    )

    raw = response.choices[0].message.content.strip()

    # Parse JSON array from the LLM response
    # Handle cases where LLM wraps in markdown code block
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    
    questions = json.loads(raw)
    elapsed = time.time() - start_time

    print(f"\n\u2705 Generated {len(questions)} questions in {elapsed:.1f}s:")
    for i, q in enumerate(questions, 1):
        print(f"  {i}. {q}")

    return questions


print("Phase 1 (Planner) ready.")

Phase 1 (Planner) ready.


In [0]:
%python
# ---------------------------------------------------------------------------
# Phase 2: BATCH QUERY — Send questions to Genie in parallel, collect data
# ---------------------------------------------------------------------------

def format_result_as_table(result: dict) -> str:
    """Convert a single Genie result into a compact text table for the data packet."""
    parts = []
    parts.append(f"### Q: {result['question']}")

    if result.get("status") == "FAILED":
        parts.append(f"**[QUERY FAILED]** {result.get('error', 'Unknown error')}")
        return "\n".join(parts)

    if result.get("text_response"):
        parts.append(f"Response: {result['text_response']}")

    cols = result.get("data_columns", [])
    rows = result.get("data_rows", [])
    if cols and rows:
        parts.append(f"Data ({len(rows)} rows):")
        parts.append(" | ".join(cols))
        parts.append(" | ".join(["---"] * len(cols)))
        for row in rows[:100]:  # Cap at 100 rows to stay within token budget
            parts.append(" | ".join(
                str(v) if v is not None else "NULL" for v in row
            ))
        if len(rows) > 100:
            parts.append(f"... and {len(rows) - 100} more rows")
    elif not result.get("text_response"):
        parts.append("**[NO DATA RETURNED]**")

    return "\n".join(parts)


def batch_query(questions: list[str], space_id: str = MARDI_SPACE_ID) -> dict:
    """Phase 2: Send all planned questions to Genie in parallel.
    
    Returns a data packet dict with:
      - raw_results: list of raw Genie response dicts
      - data_packet: formatted text ready for the synthesizer LLM
      - stats: query success/failure counts and timing
    """
    print(f"\n{'='*70}")
    print(f"PHASE 2: BATCH QUERY")
    print(f"{'='*70}")
    start_time = time.time()

    genie = GenieClient(space_id=space_id)
    raw_results = genie.batch_ask(questions)

    # Build the compact data packet for the synthesizer
    data_sections = []
    failed_queries = []
    for i, result in enumerate(raw_results):
        table_text = format_result_as_table(result)
        data_sections.append(table_text)
        if result.get("status") == "FAILED":
            failed_queries.append(i + 1)

    data_packet = "\n\n".join(data_sections)
    elapsed = time.time() - start_time
    token_estimate = len(data_packet) // 4  # Rough token estimate

    stats = {
        "total_queries": len(questions),
        "succeeded": len(questions) - len(failed_queries),
        "failed": len(failed_queries),
        "failed_indices": failed_queries,
        "elapsed_seconds": round(elapsed, 1),
        "data_packet_chars": len(data_packet),
        "data_packet_tokens_est": token_estimate,
    }

    print(f"\n\U0001f4e6 Data packet: ~{token_estimate:,} tokens ({len(data_packet):,} chars)")
    if failed_queries:
        print(f"\u26a0\ufe0f  Failed queries: {failed_queries} (report will proceed with gaps)")

    return {
        "raw_results": raw_results,
        "data_packet": data_packet,
        "stats": stats,
    }


print("Phase 2 (Batch Query) ready.")

Phase 2 (Batch Query) ready.


In [0]:
%python
# ---------------------------------------------------------------------------
# Phase 3: SYNTHESIZE — Single LLM call to produce the 7-slide report
# ---------------------------------------------------------------------------

SYNTHESIZER_SYSTEM_PROMPT = f"""
You are an expert marketing analytics report writer at Databricks.
You receive a data packet containing raw results from multiple queries about
a marketing event. Your job is to synthesize this into a structured report
following the exact slide template below.

{METRIC_DEFINITIONS}

{REPORT_TEMPLATE}

## Instructions
1. Cross-reference data across all query results to build a complete picture.
2. Calculate derived metrics where raw components are available:
   - Attendance Rate = Attended / Registered × 100
   - Cost per Lead = Total Spend / NNN
   - fPOI = U2 ARR / Allocated Spend
   - Conversion rates between funnel stages
3. For each slide, provide:
   - Headline metric(s) with actual numbers
   - Supporting data points and breakdowns
   - Key insight or takeaway (1-2 sentences)
4. If data for a slide is unavailable, note "[Data Not Available]" and explain what's missing.
5. If YoY data exists, compute percentage changes.
6. Use clean formatting with headers, bullet points, and bold for key numbers.
7. End with a brief "Data Gaps & Caveats" section noting any failed queries or missing data.
8. Be precise with numbers — do not round excessively or fabricate data.
"""


def synthesize_report(
    data_packet: str,
    user_request: str,
    stats: dict,
    llm: str = DEFAULT_LLM,
) -> str:
    """Phase 3: Send the complete data packet to an LLM to generate the report.
    
    Args:
        data_packet: Formatted text with all Genie query results
        user_request: Original user request for context
        stats: Query stats from Phase 2
        llm: Which Databricks-hosted LLM to use
    
    Returns:
        Formatted report narrative string
    """
    print(f"\n{'='*70}")
    print(f"PHASE 3: SYNTHESIZE")
    print(f"{'='*70}")
    start_time = time.time()

    endpoint = LLM_ENDPOINTS[llm.lower()]
    client = DatabricksOpenAI()

    user_message = f"""## Original Request
{user_request}

## Query Statistics
- Queries sent: {stats['total_queries']}
- Succeeded: {stats['succeeded']}
- Failed: {stats['failed']} (indices: {stats.get('failed_indices', [])})

## Raw Data from Genie
{data_packet}

Please synthesize this data into the 7-slide report format."""

    print(f"\U0001f4e4 Sending ~{stats['data_packet_tokens_est']:,} tokens to {llm.upper()} ({endpoint})...")

    response = client.chat.completions.create(
        model=endpoint,
        messages=[
            {"role": "system", "content": SYNTHESIZER_SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        temperature=0.3,
        max_tokens=4000,
    )

    report = response.choices[0].message.content
    elapsed = time.time() - start_time
    print(f"\u2705 Report synthesized in {elapsed:.1f}s")
    return report


print("Phase 3 (Synthesizer) ready.")

Phase 3 (Synthesizer) ready.


In [0]:
# ---------------------------------------------------------------------------
# Main Orchestrator — Ties all 3 phases together
# ---------------------------------------------------------------------------

def generate_event_report(
    user_request: str,
    llm: str = DEFAULT_LLM,
    space_id: str = MARDI_SPACE_ID,
) -> dict:
    """End-to-end Event ROI report generation pipeline.
    
    Phase 1: PLAN    — LLM generates 5-8 wide Genie questions (~2s)
    Phase 2: QUERY   — Parallel Genie execution (~15-20s)
    Phase 3: SYNTH   — LLM synthesizes 7-slide report (~5-10s)
    
    Args:
        user_request: Natural language like "Generate report for NRF 2026"
        llm: "claude" or "gpt" (Databricks-hosted)
        space_id: Genie space ID (defaults to MARDI)
    
    Returns:
        dict with keys:
          - report: Final synthesized report text
          - questions: List of planned Genie questions
          - data_packet: Raw data sent to synthesizer
          - stats: Timing and success/failure counts
          - raw_results: Individual Genie query responses
    """
    pipeline_start = time.time()
    print("\n" + "\u2550"*70)
    print("  EVENT ROI GENERATOR")
    print("\u2550"*70)
    print(f"Request: \"{user_request}\"")
    print(f"LLM: {llm.upper()} | Genie Space: {space_id}")
    print("\u2550"*70)

    # ----- Phase 1: PLAN -----
    questions = plan_questions(user_request, llm=llm)

    # ----- Phase 2: BATCH QUERY -----
    query_result = batch_query(questions, space_id=space_id)

    # ----- Phase 3: SYNTHESIZE -----
    report = synthesize_report(
        data_packet=query_result["data_packet"],
        user_request=user_request,
        stats=query_result["stats"],
        llm=llm,
    )

    total_elapsed = time.time() - pipeline_start
    sep = "\u2550" * 70
    print(f"\n{sep}")
    print(f"  PIPELINE COMPLETE in {total_elapsed:.1f}s")
    print(f"  Phase 1 (Plan): Generated {len(questions)} questions")
    print(f"  Phase 2 (Query): {query_result['stats']['succeeded']}/{query_result['stats']['total_queries']} succeeded in {query_result['stats']['elapsed_seconds']}s")
    print(f"  Phase 3 (Synth): Report generated")
    print(f"{sep}\n")

    return {
        "report": report,
        "questions": questions,
        "data_packet": query_result["data_packet"],
        "stats": query_result["stats"],
        "raw_results": query_result["raw_results"],
    }


print("Pipeline ready. Use generate_event_report('your request') to run.")

Pipeline ready. Use generate_event_report('your request') to run.


In [0]:
%python
# ---------------------------------------------------------------------------
# Example: Generate an Event ROI report for NRF 2026
# ---------------------------------------------------------------------------

result = generate_event_report(
    user_request="Generate a post-event ROI report for AI Days",
    llm="claude",
)

# Display the final report
print("\n" + "="*70)
print("  FINAL REPORT")
print("="*70)
print(result["report"])


══════════════════════════════════════════════════════════════════════
  EVENT ROI GENERATOR
══════════════════════════════════════════════════════════════════════
Request: "Generate a post-event ROI report for AI Days"
LLM: CLAUDE | Genie Space: 01f137f8ed74188983eabc78abf23274
══════════════════════════════════════════════════════════════════════

PHASE 1: PLAN
User request: "Generate a post-event ROI report for AI Days"

✅ Generated 8 questions in 4.7s:
  1. Show me total registered, attended, NNN, MQL, and total pipeline amount for AI Days grouped by child program
  2. What is the U2 pipeline amount, U2 ARR, U3 pipeline amount, U3 ARR, and deal count for AI Days
  3. Show the total allocated spend, cost per attendee, cost per MQL, and cost per pipeline dollar for AI Days
  4. What is the full funnel conversion from registered to attended to NNN to MQL to U2 to U3 for AI Days
  5. Show unique accounts, DBX customer vs prospect breakdown, and account segment distribution for AI Days

In [0]:
print(result["report"])

# AI Days Post-Event ROI Report

---

## Slide 1: Executive Summary

### Headline Metrics
- **Total Pipeline Created (U2):** $1,645,481 across 260.60 opportunities
- **Qualified Pipeline (U3/QRR):** $3,199,068 across 161.71 opportunities
- **Overall Attendance Rate:** 4.83% (1,516 attended / 31,394 registered)
- **Forecasted Pipeline on Investment (fPOI):** 481x ($1.65M U2 ARR / $3,421 allocated spend)

### Key Insight
AI Days generated exceptional pipeline ROI with nearly half a million times return on allocated marketing spend, though the extremely low attendance rate (4.83%) indicates significant opportunity loss in converting registrations to actual participation. The event series successfully engaged 7,000+ unique accounts across 53 programs globally, with strong pipeline maturation from U2 to U3 stages.

---

## Slide 2: Registration & Attendance

### Registration Metrics
- **Total Registered:** 31,394 across 53 child programs
- **Total Attended:** 1,516
- **Attendance Rate:** 4.